In [ ]:
import pandas as pd
import glob
from google.colab import drive

# 1. Conectar a Drive
drive.mount('/content/drive', force_remount=True)

# 2. Buscar archivos
path = '/content/drive/MyDrive/Tesis.Ecobici.2020-2025/*.csv'
files = sorted(glob.glob(path))

lista_diaria = []

if len(files) > 0:
    print(f"Procesando {len(files)} archivos con corrección de fecha...")

    for f in files:
        try:
            temp_cols = pd.read_csv(f, nrows=0).columns.tolist()
            col_fecha = 'start_date' if 'start_date' in temp_cols else ('Fecha_Retiro' if 'Fecha_Retiro' in temp_cols else None)

            if col_fecha:
                temp_df = pd.read_csv(f, low_memory=False)

                # --- AQUÍ ESTÁ LA CORRECCIÓN CLAVE ---
                # Usamos dayfirst=True y format='mixed' para que no falle con los archivos de 2020/2021
                temp_df['fecha_solo'] = pd.to_datetime(temp_df[col_fecha], dayfirst=True, errors='coerce').dt.date

                resumen = temp_df.groupby('fecha_solo').size().reset_index(name='viajes')
                lista_diaria.append(resumen)
                print(f"✅ Procesado: {f.split('/')[-1]}")
            else:
                print(f"⚠️ Saltado (sin columna de fecha): {f.split('/')[-1]}")

        except Exception as e:
            print(f"❌ Error en archivo {f.split('/')[-1]}: {e}")

    # 3. Unir y limpiar el resultado final
    if lista_diaria:
        demanda_diaria = pd.concat(lista_diaria, ignore_index=True)
        demanda_diaria = demanda_diaria.groupby('fecha_solo')['viajes'].sum().reset_index()
        demanda_diaria.rename(columns={'fecha_solo': 'fecha'}, inplace=True)

        demanda_diaria['fecha'] = pd.to_datetime(demanda_diaria['fecha'])
        demanda_diaria['dia_semana'] = demanda_diaria['fecha'].dt.dayofweek
        demanda_diaria['mes'] = demanda_diaria['fecha'].dt.month
        demanda_diaria['anio'] = demanda_diaria['fecha'].dt.year

        print("\n🚀 ¡ÉXITO! Ahora sí procesamos todos los meses sin errores de formato.")
        print(demanda_diaria.head())
    else:
        print("❌ No se pudo procesar nada.")


Mounted at /content/drive
Procesando 73 archivos con corrección de fecha...
⚠️ Saltado (sin columna de fecha): dashboard_ecobici_final.csv
✅ Procesado: ecobici_2020_01.csv
✅ Procesado: ecobici_2020_02.csv
✅ Procesado: ecobici_2020_03.csv
✅ Procesado: ecobici_2020_04.csv
✅ Procesado: ecobici_2020_05.csv
✅ Procesado: ecobici_2020_06.csv
✅ Procesado: ecobici_2020_07.csv
✅ Procesado: ecobici_2020_08.csv
✅ Procesado: ecobici_2020_09.csv
✅ Procesado: ecobici_2020_10.csv
✅ Procesado: ecobici_2020_11.csv
✅ Procesado: ecobici_2020_12.csv
✅ Procesado: ecobici_2021_01.csv
✅ Procesado: ecobici_2021_02.csv
✅ Procesado: ecobici_2021_03.csv
✅ Procesado: ecobici_2021_04.csv
✅ Procesado: ecobici_2021_05.csv
✅ Procesado: ecobici_2021_06.csv
✅ Procesado: ecobici_2021_07.csv
✅ Procesado: ecobici_2021_08.csv
✅ Procesado: ecobici_2021_09.csv
✅ Procesado: ecobici_2021_10.csv
✅ Procesado: ecobici_2021_11.csv
✅ Procesado: ecobici_2021_12.csv
✅ Procesado: ecobici_2022_01.csv
✅ Procesado: ecobici_2022_02.csv
✅ P

In [ ]:
# --- NUEVA CELDA: VARIABLES LAG ---

# Ordenar por fecha
demanda_diaria = demanda_diaria.sort_values('fecha')

# Crear lags
demanda_diaria['lag_1'] = demanda_diaria['viajes'].shift(1)
demanda_diaria['lag_7'] = demanda_diaria['viajes'].shift(7)

# Eliminar nulos
demanda_diaria = demanda_diaria.dropna()

print("✅ Lags creados")
print(demanda_diaria.head())

✅ Lags creados
        fecha  viajes  dia_semana  mes  anio   lag_1  lag_7
7  2019-12-29       2           6   12  2019     1.0    1.0
8  2019-12-30       3           0   12  2019     2.0    1.0
9  2019-12-31      26           1   12  2019     3.0    1.0
10 2020-01-01    3854           2    1  2020    26.0    1.0
11 2020-01-02   16374           3    1  2020  3854.0    1.0


In [ ]:
# --- CELDA DE ENTRENAMIENTO DE MODELOS ---
from sklearn.ensemble import RandomForestRegressor
from statsmodels.tsa.ar_model import AutoReg

# 1. Preparar datos para Random Forest (RF)
X = demanda_diaria[['dia_semana', 'mes', 'anio', 'lag_1', 'lag_7']]
y = demanda_diaria['viajes']

# Crear y entrenar Random Forest
modelo_rf = RandomForestRegressor(n_estimators=100, random_state=42)
modelo_rf.fit(X, y)
print("✅ Modelo Random Forest entrenado.")

# 2. Entrenar Modelo Autorregresivo (AR)
# Usamos un lag de 7 días (una semana)
modelo_ar_fit = AutoReg(y, lags=7).fit()
# Obtenemos las predicciones para todo el conjunto
modelo_ar = modelo_ar_fit.predict(start=0, end=len(y)-1).values
print("✅ Modelo AR entrenado.")


✅ Modelo Random Forest entrenado.
✅ Modelo AR entrenado.


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)


In [ ]:
# --- PASO 4: LIMPIEZA FINAL, PREDICCIONES Y EXPORTACIÓN ---

# ⚠️ IMPORTANTE: usar las MISMAS variables con las que entrenaste el modelo
X = demanda_diaria[['dia_semana', 'mes', 'anio', 'lag_1', 'lag_7']]

# 1. Predicciones
demanda_diaria['rf_pred'] = modelo_rf.predict(X)
demanda_diaria['ar_pred'] = modelo_ar

# 2. Cálculo de errores
demanda_diaria['error_rf'] = abs(demanda_diaria['viajes'] - demanda_diaria['rf_pred'])
demanda_diaria['error_ar'] = abs(demanda_diaria['viajes'] - demanda_diaria['ar_pred'])

# 3. Filtrar solo desde 2020
demanda_diaria = demanda_diaria[demanda_diaria['anio'] >= 2020].copy()

# 🔥 4. ELIMINAR VALORES FALTANTES (CORRECCIÓN FINAL)
demanda_diaria = demanda_diaria.dropna()

# 5. Exportar a Drive
ruta_final = '/content/drive/MyDrive/Tesis.Ecobici.2020-2025/dashboard_ecobici_final.csv'
demanda_diaria.to_csv(ruta_final, index=False)

# 6. Validación final
print("✅ ¡Archivo final limpio y listo para Power BI!")
print(f"📊 Año inicial: {demanda_diaria['anio'].min()}")
print(f"📈 Filas totales: {len(demanda_diaria)}")
print(f"📉 Valores nulos por columna:\n{demanda_diaria.isnull().sum()}")
print(demanda_diaria.tail())

✅ ¡Archivo final limpio y listo para Power BI!
📊 Año inicial: 2020
📈 Filas totales: 2163
📉 Valores nulos por columna:
fecha         0
viajes        0
dia_semana    0
mes           0
anio          0
lag_1         0
lag_7         0
rf_pred       0
ar_pred       0
error_rf      0
error_ar      0
dtype: int64
          fecha  viajes  dia_semana  mes  anio    lag_1    lag_7   rf_pred  \
2168 2025-12-20   37341           5   12  2025  52744.0  38501.0  37261.86   
2169 2025-12-21   30412           6   12  2025  37341.0  35937.0  31303.62   
2170 2025-12-22   42664           0   12  2025  30412.0  56346.0  46983.65   
2171 2025-12-23   44962           1   12  2025  42664.0  60858.0  48119.50   
2172 2025-12-24   28172           2   12  2025  44962.0  54208.0  38090.83   

           ar_pred  error_rf     error_ar  
2168  36637.039496     79.14   703.960504  
2169  28559.227302    891.62  1852.772698  
2170  40147.585086   4319.65  2516.414914  
2171  42317.340686   3157.50  2644.659314  
2172